# HW4 - Notebook 3: Backtester accounting and timing checks

Checks on [`backtester.py`](backtester.py) before the final test: hand checked toy markets, a dry run on 2012–2015
audited on every row, and one trade recomputed from SQL.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
import sys

# Course setup
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
modeling_dir = course_root / 'Lectures/modeling'
if str(modeling_dir) not in sys.path:
    sys.path.insert(0, str(modeling_dir))

import duckdb
from data import DATABASE
from features import PANEL_PATH
from model import investable, walk_forward, yearly_fit_dates, candidates, FINAL_MODEL, TOP_N
from backtester import plan_picks, load_price_matrix, run_backtest, check_accounting, FEE_RATE

## Trading rules

| Rule | Implementation |
|---|---|
| Capital | \$1,000,000, long only, no borrowing |
| Timing | Signal at month-end close, fill at the next session's close |
| Picks | Top 30 by factor score, equal weight |
| Batch size | `min(cash, 1/3 × portfolio value)` per month |
| Exit | 90 calendar days later, or the next session with a quote |
| Costs | 10 bps per buy and sell; cash earns no interest |
| Missing quotes | Entry: skip the buy. Holding: last close. No quote within 5 sessions after maturity: written off at \$0 |

## 1. Toy market checked by hand
A tiny madeup market for hand check: \$1,000 buys A (\$10) and B (\$20) on Mon Jan 6, 2020.
A rises to \$12, B stops trading after Feb 14. Fee = 0.1% per side.

In [ ]:
# weekday sessions only
sessions = pd.bdate_range('2020-01-01', '2020-06-30')
toy = pd.DataFrame({'A': 10.0, 'B': 20.0}, index=sessions)
# A rises to 12 from Mar 2
toy.loc['2020-03-02':, 'A'] = 12.0
# B stops trading after Feb 14
toy.loc['2020-02-17':, 'B'] = np.nan
# signal Friday Jan 3
toy_picks = pd.Series({pd.Timestamp('2020-01-03'): ['A', 'B']})
r = run_backtest(toy, toy_picks, '2020-01-06', '2020-06-30', initial_cash=1000, batch_fraction=1.0)
buys = r.ledger[r.ledger.type == 'buy'].set_index('symbol')
lots = r.lots.set_index('symbol')

# Entry Mon Jan 6. 500 each: A shares = 500/(10 x 1.001) = 49.950050, fee = 0.499500
assert (buys.date == pd.Timestamp('2020-01-06')).all()
assert np.isclose(buys.loc['A', 'shares'], 49.950050) and np.isclose(buys.loc['A', 'fee'], 0.499500)
# Mar 2: A = 49.950050 x 12 = 599.400599. B stale at 20: 24.975025 x 20 = 499.500500
assert np.isclose(r.history.loc['2020-03-02', 'value'], 1098.901099)
assert np.allclose(r.history.value, r.history.cash + r.history.positions_value)
# Jan 6 + 90 days = Sunday Apr 5. Sold Monday Apr 6 for 599.400599 x 0.999 = 598.801199
assert lots.loc['A', 'exit_date'] == pd.Timestamp('2020-04-06') and np.isclose(lots.loc['A', 'proceeds'], 598.801199)
# B: no quote on Apr 6 or the 5 sessions after, so write off at $0 on Apr 13
assert lots.loc['B', 'exit_type'] == 'write_off' and lots.loc['B', 'exit_date'] == pd.Timestamp('2020-04-13')
assert np.isclose(r.history.cash.iloc[-1], 598.801199)

# Sensitivity: sell B at its last price instead: 499.500500 x 0.999 = 499.000999
b = run_backtest(toy, toy_picks, '2020-01-06', '2020-06-30', initial_cash=1000, batch_fraction=1.0,
                 missing_exit='last_price').lots.set_index('symbol').loc['B']
assert b.exit_type == 'exit_at_last_price' and np.isclose(b.proceeds, 499.000999)
display(lots[['entry_date', 'shares', 'exit_date', 'exit_price', 'proceeds', 'exit_type']])
print('Toy market matches the hand calculations.')

,entry_date,shares,exit_date,exit_price,proceeds,exit_type
symbol,,,,,,
A,2020-01-06,49.950050,2020-04-06,12.0,598.801199,sell
B,2020-01-06,24.975025,2020-04-13,0.0,0.000000,write_off


Toy market matches the hand calculations.


## 2. Dry run on 2012–2015 and full audit

In [ ]:
# select eligible stocks
panel = investable(pd.read_parquet(PANEL_PATH))
pipeline, features, target = candidates[FINAL_MODEL]
# dry run stops at the end of the development period
DEV_END = '2015-12-31'

# score every stock at each month-end 2012-2015, and record each fit and data used
scores, audit = walk_forward(panel, pipeline, features, target,
                             yearly_fit_dates(panel, '2012-01-01', DEV_END), end=DEV_END)
# each fit only uses outcomes finished before its fit date
display(audit)

# picks top 30 stocks of each month 
picks = plan_picks(scores, TOP_N)
# loads daily price 
prices, bench = load_price_matrix(np.concatenate(picks.values), end=DEV_END)
# runs the backtest from the 1st trading day through 2015
start = prices.index[prices.index.get_indexer([picks.index[0]])[0] + 1]
result = run_backtest(prices, picks, start=start, end=DEV_END)
print(f'{len(result.lots):,} purchases; exits: {result.lots.exit_type.value_counts().to_dict()}')
display(result.history.head(3))

,fit_date,training_rows,last_finished_outcome,scores_signals,scored_rows
0,2012-01-31,28553,2012-01-30,2012-01-31 .. 2012-12-31,16210
1,2013-01-31,44701,2013-01-30,2013-01-31 .. 2013-12-31,17905
2,2014-01-31,62195,2014-01-30,2014-01-31 .. 2014-12-31,20038
3,2015-01-30,80272,2014-12-30,2015-01-30 .. 2015-12-31,21199


1,380 purchases; exits: {'sell': 1320, 'open_at_end': 60}


,cash,positions_value,value,open_lots,stale_lots
date,,,,,
2012-02-01,666666.666667,333000.333000,9.996670e+05,30,0
2012-02-02,666666.666667,331714.653410,9.983813e+05,30,0
2012-02-03,666666.666667,335341.728765,1.002008e+06,30,0


Day one: \$333,333.33 is spent (1/3 of \$1M) on 30 stocks including \$333 of fees, so the value is \$999,667.

In [ ]:
# checks for accounting
# every check must be True
display(check_accounting(result, prices, DEV_END))

,passed
"value = cash + positions, every day",True
cash never negative,True
final cash = start - buys - fees + sale proceeds,True
each batch budget <= cash at its signal,True
every buy fills at the session after its signal,True
entry price = that day's adjusted close,True
every closed lot held >= 90 calendar days,True
sells happen at the first quote on/after maturity,True
exit price = that day's adjusted close,True
no trade or value after the end date,True


## 3. One real trade recomputed from SQL

In [ ]:
# Take the first purchase of the run and its budget. The first batch split evenly over its 30 stocks
lot = result.lots.iloc[0]
budget = result.orders.budget.iloc[0] / result.orders.picks.iloc[0]

# Read this stock's prices straight from the database without using the backtester
with duckdb.connect(str(DATABASE), read_only=True) as con:
    raw = con.execute('SELECT date, adj_close FROM prices WHERE symbol = ? AND date BETWEEN ? AND ? ORDER BY date',
                      [lot.symbol, lot.entry_date, lot.exit_date]).fetchdf()
raw = raw.set_index(pd.to_datetime(raw.date)).adj_close

# do the trade by hand
# 90 days after entry
due = lot.entry_date + pd.Timedelta(days=90)
# sell on the first trading day on/ after the due date
exit_date = raw.index[raw.index >= due][0]
shares = budget / (raw[lot.entry_date] * (1 + FEE_RATE))
proceeds = shares * raw[exit_date] * (1 - FEE_RATE)

# Compare hand calculation and the backtester
display(pd.DataFrame({
    'by hand': [lot.symbol, lot.entry_date.date(), exit_date.date(), round(shares, 6), round(proceeds, 2)],
    'backtester': [lot.symbol, lot.entry_date.date(), lot.exit_date.date(), round(lot.shares, 6), round(lot.proceeds, 2)]},
    index=['symbol', 'entry date', 'exit date', 'shares', 'proceeds after fee']))
assert exit_date == lot.exit_date and np.isclose(shares, lot.shares) and np.isclose(proceeds, lot.proceeds)
print('Hand calculation matches the backtester.')

,by hand,backtester
symbol,FUN,FUN
entry date,2012-02-01,2012-02-01
exit date,2012-05-01,2012-05-01
shares,725.965409,725.965409
proceeds after fee,13228.37,13228.37


Hand calculation matches the backtester.
